# Sleep Time Prediction Project

This notebook explores how common lifestyle habits relate to sleep duration and uses a simple K-nearest neighbors regression model to predict `SleepTime`.


### Introduction
Sleep is essential for physical and mental health, yet many college
students struggle to get enough of it. For us students, balancing academics and lifestyle habits and understanding how these everyday behaviours relate to our sleep duration
can help us identify which habits to adjust in order to improve our sleep,
maintain productivity, and support our overall wellbeing.

The dataset used in this project was obtained from [Kaggle](https://www.kaggle.com/datasets/govindaramsriram/sleep-time-prediction/data). It includes 2,000
observations of daily lifestyle measurements — workout time, reading time, phone time, work hours, caffeine intake, and relaxation time. This dataset was generated synthetically but for our exploration, it provides a useful basis for analyzing how lifestyle behaviours are
associated with sleep duration.

Given that our dataset contains six numerical lifestyle features and a continuous sleep duration target, this dataset is well-suited for our analysis, as all features and the target variable are numerical, making it compatible with a correlation matrix and KNN regression model.

We want to explore these questions:

- Which daily lifestyle habits are most related to sleep duration?
- Which daily lifestyle habits best predict sleep duration?


Team Members:

- Hazel Delos Reyes
- Praneel Talluri


In [ ]:
import pandas as pd
import altair as alt
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

alt.data_transformers.enable("vegafusion")

df = pd.read_csv("data/sleeptime_prediction_dataset.csv")
df.head()


First, we look at whether the lifestyle habits in the dataset are correlated with sleep time. This helps us see which variables appear most related to the target before building a predictive model.


In [3]:
sleep_df = df.drop(columns=["SleepTime"])
sleep_df.head()

,WorkoutTime,ReadingTime,PhoneTime,WorkHours,CaffeineIntake,RelaxationTime
0,1.12,0.52,3.29,7.89,216.08,0.75
1,2.85,0.49,4.22,5.03,206.18,0.67
2,2.20,1.81,4.04,9.23,28.73,0.35
3,1.80,0.50,1.62,7.68,276.77,1.21
4,0.47,0.54,1.60,4.94,170.54,0.95


The features used in this analysis are `WorkoutTime`, `ReadingTime`, `PhoneTime`, `CaffeineIntake`, `RelaxationTime`, and `WorkHours`. While most of these represent personal habits, `WorkHours` is also included because academic and work demands can directly compete with time available for sleep.


In [4]:
# Analyze Variables by plotting them; Look for correlations between variables; correlation matrix
# plot have captions, are explained with enough information

corr_matrix = df.corr(numeric_only=True).reset_index().melt('index')

col_order = list(df.select_dtypes('number').columns)

alt.Chart(corr_matrix).mark_rect().encode(
    x=alt.X('index:N', title=None, sort=col_order, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('variable:N', title=None, sort=col_order),
    color=alt.Color('value:Q',
        title='Sleep Correlation',
        scale=alt.Scale(scheme='redblue', reverse=True, domain=[-1, 1])
    )
).properties(
    title='Correlation Heatmap of Lifestyle Factors',
    width=400,
    height=400
)


alt.Chart(...)

Here, we use a correlation heatmap to explore which lifestyle habits are related to sleep duration. The color scale makes negative and positive correlations easier to compare across variables.


In [5]:
df.corr(numeric_only=True)['SleepTime'].sort_values()

PhoneTime        -0.322506
WorkHours        -0.298469
CaffeineIntake   -0.076992
ReadingTime       0.067199
RelaxationTime    0.144243
WorkoutTime       0.188368
SleepTime         1.000000
Name: SleepTime, dtype: float64

Phone Time (-0.32) and Work Hours (-0.30) show the strongest relationships with sleep duration. Both are moderate negative correlations, meaning that increased phone use and longer work hours are associated with less sleep.

Workout Time (+0.19) and Relaxation Time (+0.14) show weaker positive relationships, suggesting these activities may be associated with slightly longer sleep duration.

Reading Time (+0.07) and Caffeine Intake (-0.08) show very weak relationships with sleep duration. Overall, no single habit strongly predicts sleep by itself, so a model that combines multiple variables may be more useful.


### Predictive Modeling

Because `SleepTime` is a numerical value, this is a supervised regression problem. We will use the six lifestyle variables to predict sleep duration.

We use K-nearest neighbors regression because it is a simple model from class that can make predictions based on similar observations. Since KNN depends on distance, we scale the predictors so that variables with larger numeric ranges, such as `CaffeineIntake`, do not dominate the distance calculation.


In [ ]:
predictors = [
    "WorkoutTime",
    "ReadingTime",
    "PhoneTime",
    "WorkHours",
    "CaffeineIntake",
    "RelaxationTime",
]

X = df[predictors]
y = df["SleepTime"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.75, random_state=2020
)

knn_pipe = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor()
)

param_grid = {"kneighborsregressor__n_neighbors": range(1, 101)}

knn_tune = GridSearchCV(
    knn_pipe,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

knn_tune.fit(X_train, y_train)

best_k = knn_tune.best_params_["kneighborsregressor__n_neighbors"]
cv_rmse = -knn_tune.best_score_

best_k, cv_rmse


The cross-validation step compares different values of `k`, which controls how many nearby observations are used to make each prediction. A very small `k` can follow noise too closely, while a very large `k` can smooth over useful patterns. We choose the value with the lowest cross-validation RMSE.


In [ ]:
best_model = knn_tune.best_estimator_
best_model


### Model Evaluation

The final step is to test the selected model on observations that were not used during training or cross-validation. This gives a more honest estimate of how well the model predicts sleep duration for new people.


In [ ]:
sleep_predictions = best_model.predict(X_test)

test_rmspe = mean_squared_error(y_test, sleep_predictions) ** 0.5

baseline_predictions = [y_train.mean()] * len(y_test)
baseline_rmspe = mean_squared_error(y_test, baseline_predictions) ** 0.5

model_results = pd.DataFrame({
    "Model": ["KNN regression", "Training mean baseline"],
    "RMSPE (hours)": [test_rmspe, baseline_rmspe]
})

model_results


In [ ]:
prediction_preview = pd.DataFrame({
    "Actual SleepTime": y_test.reset_index(drop=True).head(10),
    "Predicted SleepTime": sleep_predictions[:10]
})

prediction_preview


### Conclusion

The exploratory analysis suggests that no single lifestyle habit completely explains sleep duration. The strongest relationships with `SleepTime` are `PhoneTime` and `WorkHours`, both of which have moderate negative correlations. This means people who spend more time on their phones or working tend to sleep less in this dataset. `WorkoutTime` and `RelaxationTime` have smaller positive relationships with sleep, while `ReadingTime` and `CaffeineIntake` have very weak relationships.

The KNN regression model used all six lifestyle variables together to predict sleep duration. Cross-validation selected `k = 61`, meaning each prediction is based on the 61 most similar training observations after scaling the predictors. On the test data, the model has an RMSPE of about 1.71 hours. This is better than predicting the training-set average sleep time for everyone, but the error is still fairly large.

Overall, the model can identify some useful patterns, but it should not be treated as a highly precise sleep predictor. The results support the idea that sleep duration is influenced by multiple habits at the same time, with phone use and work hours appearing most strongly related to less sleep in this dataset.
